### 1. 配置vLLM环境

In [2]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

Config set: http://localhost:8000/v1
{"object":"list","data":[{"id":"Qwen3.5-9B","object":"model","created":1793438678,"owned_by":"vllm","root":"/root/tmp/agents-101/modelscope/models/Qwen--Qwen3.5-9B/snapshots/master","parent":null,"max_model_len":8192,"permission":[{"id":"modelperm-ac5e4aa3df1162dc","object":"model_permission","created":1793438678,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

### 2. 创建基于pydantic的Agent

In [3]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("Qwen3.5-9B", provider=provider)
from pydantic_ai import Agent

agent = Agent[None, str](
    model=agent_model
)

In [4]:
# 准备已有的MCP服务 + 封装异步函数: agent.run
import asyncio
from pydantic_ai.mcp import MCPServerStdio
async def run_async(prompt: str) -> str:
    # (1) 加载已有的MCP服务
    async with agent.run_mcp_servers():
        # （2）agent.run本身就是异步函数。
        result = await agent.run(prompt)
        return result.output
await run_async("What is the capital of France?")

'\n\nThe capital of France is **Paris**.'

### 3. 给大模型增加上MCP工具

In [5]:
#  模型本身没有时间权限
await run_async("What’s the date today?")

"\n\nI don't have access to real-time data, so I can't provide today's exact date. Please check your device or calendar for the current date! 📅"

In [6]:
# 给模型本地的datetime服务
from pydantic_ai.agent import Agent


from datetime import datetime
from pydantic_ai import Tool          
@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime(format="%Y-%m-%d %H:%M:%S")

agent: Agent[None, str] = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)
await run_async("What’s the date today?")

"\n\nToday's date is **October 31, 2026**.\n\nThe current time is 5:25:26 PM."

In [7]:
# 给模型在线mcp服务
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=America/New_York",
    ],
)
agent = Agent(
    model=agent_model,
    mcp_servers=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

# 调用封装好的工具
answer = await run_async(
    "Use the get_current_time tool to find today's date "
    "in the America/New_York timezone."
)
print(answer)

# 查看模型输出明细
async with agent.run_mcp_servers():
    result = await agent.run(
        "Use get_current_time to find today's date "
        "in the America/New_York timezone."
    )
print("最终回答：", result.output)
print("\n完整消息：")
for message in result.all_messages():
    print(message)



Here's today's date and time in the America/New_York timezone:

**Date:** Saturday, October 31, 2026  
**Time:** 05:25:45 AM  
**Timezone:** America/New_York (UTC-4, EST)  

Daylight Saving Time is currently in effect in this timezone.
最终回答： 

Today's date in the America/New_York timezone is **Saturday, October 31, 2026**.

The current time is 5:25:55 AM, and the timezone is currently observing Daylight Saving Time (UTC-4).

完整消息：
ModelRequest(parts=[SystemPromptPart(content='You are a helpful agent and you have access to this tool:\n   get_current_time(params: dict)\nWhen the user asks for the current date or time, call get_current_time.\n', timestamp=datetime.datetime(2026, 10, 31, 9, 25, 52, 852290, tzinfo=datetime.timezone.utc)), UserPromptPart(content="Use get_current_time to find today's date in the America/New_York timezone.", timestamp=datetime.datetime(2026, 10, 31, 9, 25, 52, 852297, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 10, 31, 9, 25, 52, 85246

### 4. 给大模型指定功能

In [31]:
!node -v && npm -v && npx --version
!mkdir -p /tmp/airbnb-mcp
!npm install --prefix /tmp/airbnb-mcp @openbnb/mcp-server-airbnb

v22.23.2
10.9.8
10.9.8
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
up to date, audited 127 packages in 2s
⠧
⠧56 packages are looking for funding
⠧  run `npm fund` for details
⠧
found 0 vulnerabilities
⠧

In [ ]:
from pathlib import Path

airbnb_entry: Path = Path("/tmp/airbnb-mcp/node_modules/@openbnb/mcp-server-airbnb/dist/index.js")

# Increase the Photon timeout to tolerate slow network responses.
source = airbnb_entry.read_text()
old_timeout = "const timeoutId = setTimeout(() => controller.abort(), 5000);"
new_timeout = "const timeoutId = setTimeout(() => controller.abort(), 30000);"
assert source.count(old_timeout) + source.count(new_timeout) == 1
airbnb_entry.write_text(source.replace(old_timeout, new_timeout))


# time_server = MCPServerStdio(
#     "python",
#     args=[
#         "-m", "mcp_server_time",
#         "--local-timezone=America/New_York",
#     ],
# )


airbnb_server = MCPServerStdio(
    "node", args=[str(airbnb_entry), "--ignore-robots-txt"],
    timeout=60,
    env={
        **{key: value for key, value in os.environ.items()
           if key.lower() in {"http_proxy", "https_proxy", "all_proxy", "no_proxy"}},
        "NODE_USE_ENV_PROXY": "1",
    },
)

'\n\nI found several great options for you to stay in Vancouver on December 1, 2026! Here are some top-rated listings:\n\n## **Top Recommendations:**\n\n### 1. **5-Star Yaletown King Room** ⭐⭐⭐⭐⭐\n- **Price:** ¥1,328 CNY/night\n- **Rating:** 5.0/5 (43 reviews)\n- **Features:** King bed, Office, Parking, AC\n- **Location:** Yaletown (49.27973, -123.12022)\n- **Book here:** [Link](https://www.airbnb.com/rooms/1684339098634343824)\n\n### 2. **Luxury High-Ceiling Suite** ⭐⭐⭐⭐⭐\n- **Price:** ¥806 CNY/night\n- **Rating:** 4.97/5 (181 reviews)\n- **Features:** 2 beds, High ceilings, Spacious, Free parking\n- **Location:** Southwest Vancouver\n- **Book here:** [Link](https://www.airbnb.com/rooms/1414824281862609969)\n\n### 3. **Cozy Studio with Private Yard** ⭐⭐⭐⭐⭐\n- **Price:** ¥516 CNY/night\n- **Rating:** 5.0/5 (53 reviews)\n- **Features:** Renovated studio, Private yard, AC\n- **Location:** East Vancouver\n- **Book here:** [Link](https://www.airbnb.com/rooms/1701906286392762447)\n\n### 4. 

In [9]:
from IPython.display import display, Markdown

system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
When the user asks for listings, first call get_current_time, then airbnb_search, etc.
"""

agent = Agent(
    model=agent_model,
    mcp_servers=[time_server, airbnb_server],
    system_prompt=system_prompt,
)

answer = await run_async(
    "Find a place to stay in Vancouver for next Sunday "
    "for 3 nights for 2 adults. "
    "Return clean Markdown with valid links."
)

Path("vancouver_stays.md").write_text(answer, encoding="utf-8")
display(Markdown(answer))



I found 18 amazing Vancouver stays for you! Here are the best options for 3 nights (Nov 1-4, 2026) for 2 adults:

| # | Property | Location | Rating | Price (3 nights) | [Book Now](https://www.airbnb.com/rooms/[ID]) |
|---|----------|----------|--------|------------------|---------------------------------------------|
| 1 | 🏡 **Evening Seattle Stay** | Downtown | New | ¥2,832 | [View →](https://www.airbnb.com/rooms/1772890227651152439) |
| 2 | 🏢 D/Town Vancouver Condo Near Granville | Downtown | ⭐ 4.75 | ¥1,624 | [View →](https://www.airbnb.com/rooms/842904285830420573) |
| 3 | 🌟 Prime DT 2BR/2BA Steps to Robson | Downtown | ⭐ 4